In [ ]:
#| echo: false
%load_ext autoreload
%autoreload 2

# Profile forecasting stages
> Find where MLForecast spends time and memory during fit and cross validation

Pass a `Profiler` through `hooks` to record the nested stages of every call.

In [ ]:
import lightgbm as lgb
from sklearn.linear_model import LinearRegression

from mlforecast import MLForecast
from mlforecast.hooks import Profiler
from mlforecast.utils import generate_daily_series

In [ ]:
series = generate_daily_series(10, min_length=100, max_length=200)

Setting `memory=True` also records the change in the process' resident memory across each stage, which requires psutil.

In [ ]:
profiler = Profiler(memory=True)
fcst = MLForecast(
    models={'lr': LinearRegression(), 'lgb': lgb.LGBMRegressor(verbosity=-1)},
    freq='D',
    lags=[1, 7],
    hooks=[profiler],
)
cv = fcst.cross_validation(series, n_windows=3, h=7)
spans = profiler.to_df()
spans[['stage', 'depth', 'seconds', 'rss_delta_bytes', 'model']]

To compare model fit costs, select only the `fit_model` rows:

In [ ]:
spans.query("stage == 'fit_model'").groupby('model')['seconds'].sum()

The `predict_model` rows do the same for prediction. They cover each model's feature updates as well as its `predict` calls:

In [ ]:
spans.query("stage == 'predict_model'").groupby('model')['seconds'].sum()

Each stage's duration includes its child stages, so summing every row double counts. Failed stages remain in the table with an `error` value. To profile another run, call `profiler.reset()` after the first run finishes.